In [1]:
%pip install qdrant-client 
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document

# Load environment variables
load_dotenv()

# connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True
)

In [3]:
# Criar coleção
client.create_collection(
    collection_name="livros",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

True

In [4]:
catalogo_livros = [
    ("1984", "Um romance distópico sobre um regime totalitário que vigia constantemente seus cidadãos e controla a verdade.", "Ficção Científica", "George Orwell"),
    ("O Senhor dos Anéis", "Uma jornada épica de um grupo de heróis para destruir um anel mágico e salvar a Terra Média do mal.", "Fantasia", "J.R.R. Tolkien"),
    ("Orgulho e Preconceito", "A história das irmãs Bennet na Inglaterra do século XIX, focando nas questões de casamento, moralidade e status social.", "Romance", "Jane Austen"),
    ("Duna", "Intrigas políticas, traições e religião em um planeta deserto que é a única fonte da substância mais valiosa do universo.", "Ficção Científica", "Frank Herbert"),
    ("O Código Da Vinci", "Um detetive e uma criptógrafa investigam um assassinato no Louvre que os leva a um segredo religioso milenar.", "Suspense", "Dan Brown"),
    ("Sapiens", "Uma exploração profunda e fascinante sobre a história da humanidade, desde a Idade da Pedra até o século XXI.", "História", "Yuval Noah Harari"),
    ("O Iluminado", "Um escritor e sua família ficam isolados durante o inverno em um hotel assombrado que o leva à loucura.", "Terror", "Stephen King"),
    ("A Revolução dos Bichos", "Animais de uma fazenda se rebelam contra seus donos humanos, mas logo criam uma nova sociedade opressiva.", "Ficção/Sátira", "George Orwell"),
    ("Pai Rico, Pai Pobre", "Lições fundamentais sobre dinheiro, investimentos e como alcançar a independência financeira.", "Finanças", "Robert Kiyosaki"),
    ("O Pequeno Príncipe", "Um piloto perdido no deserto encontra um jovem príncipe de outro planeta, trazendo reflexões sobre a vida e a natureza humana.", "Infantil/Filosofia", "Antoine de Saint-Exupéry")
]

# Gerador de pontos
points = []
for i, livro in enumerate(catalogo_livros):
    point = PointStruct(
        id=i,
        vector=Document(
            # Combinamos o título e a sinopse para o modelo gerar o contexto
            text=f"{livro[0]}: {livro[1]}", 
            model="sentence-transformers/all-MiniLM-L6-v2"
        ),
        payload={
            "titulo": livro[0],
            "sinopse": livro[1],
            "genero": livro[2],
            "autor": livro[3],
        }
    )
    points.append(point)

# Fazer o upsert (inserir) dos pontos na coleção
client.upsert(
  collection_name="livros",
  points=points,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
# Nossa busca baseada no "significado"
query_text = "sociedade opressiva e controle do governo"

# Buscar pelos livros mais similares
results = client.query_points(
    collection_name="livros",
    query=Document(text=query_text, model="sentence-transformers/all-MiniLM-L6-v2"),
    with_payload=True,
    limit=3 # Queremos apenas os 3 melhores resultados
)

# Imprimir os resultados
print(f"Buscando por: '{query_text}'\n")
for result in results.points:
    print(f"Livro: {result.payload.get('titulo', 'N/A')}")
    print(f"Autor: {result.payload.get('autor', 'N/A')}")
    print(f"Score: {result.score:.4f}")
    print(f"Sinopse: {result.payload['sinopse'][:100]}...")
    print("---")

Buscando por: 'sociedade opressiva e controle do governo'

Livro: 1984
Autor: George Orwell
Score: 0.5843
Sinopse: Um romance distópico sobre um regime totalitário que vigia constantemente seus cidadãos e controla a...
---
Livro: A Revolução dos Bichos
Autor: George Orwell
Score: 0.4907
Sinopse: Animais de uma fazenda se rebelam contra seus donos humanos, mas logo criam uma nova sociedade opres...
---
Livro: Duna
Autor: Frank Herbert
Score: 0.4722
Sinopse: Intrigas políticas, traições e religião em um planeta deserto que é a única fonte da substância mais...
---
